# Yahoo Fantasy Baseball 

Thin notebook — all logic lives in `yahoo.py`.
Uses the `Yahoo` class and helper functions from `yahoo.py` to manage your fantasy team via the Yahoo Fantasy Sports API.

**Credentials required in `.env`:**
```
Client_ID:      "your_yahoo_app_client_id"
Client_Secret:  "your_yahoo_app_client_secret"
Yahoo_Email:    "your_yahoo_account_email"
Yahoo_Password: "your_yahoo_account_password"
```

Section 1 handles OAuth automatically — no manual steps needed.  
On subsequent runs, saved tokens are reused automatically.

## Section 1: Setup & Auth

In [37]:
import importlib
import logging
import sys
import os

# Add project root to path before importing local modules
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import pandas as pd
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

logging.getLogger('yahoo_oauth').setLevel(logging.WARNING)

import yahoo as _yahoo_mod
importlib.reload(_yahoo_mod)
from yahoo import Yahoo, init_auth, top_available_all_leagues, all_rosters, top_available_with_stats, upgrade_candidates

SEASON     = 2026
CREDS_FILE = init_auth()  # reads .env, handles OAuth, returns creds_file path

Credentials written to /Users/msands/Library/CloudStorage/OneDrive-Personal/Documents/code/baseball/browser/yahoo_oauth.json — tokens present
Refresh token found — no browser needed.
OAuth OK — session ready


## Section 2: Find Your League ID

Lists all your active MLB fantasy leagues. Copy the `league_id` for your team into the config cell below.

In [38]:
leagues_df = Yahoo.list_leagues(CREDS_FILE, SEASON)
print(f'Found {len(leagues_df)} MLB league(s) for {SEASON}:')
leagues_df

Found 10 MLB league(s) for 2026:


,league_id,team_name,name,scoring_method,num_teams,draft_status,league_key
0,18397,Mother Tuckers 100,Yahoo Prize H2H-Cat 18397,categories,12,postdraft,469.l.18397
1,41036,Dr. StrangeGlove 100,Yahoo Prize H2H-Cat 41036,categories,12,postdraft,469.l.41036
2,15981,Suave 100,Yahoo Prize Roto 15981,roto,12,postdraft,469.l.15981
3,68331,Japanese Kokamo Okamo 100,Yahoo Prize H2H-Cat 68331,categories,12,postdraft,469.l.68331
4,148341,Schlittler 100,Yahoo Prize H2H-Cat 148341,categories,12,postdraft,469.l.148341
5,3924,My Bradish Burns 100,Yahoo Prize H2H-Cat 3924,categories,12,postdraft,469.l.3924
6,690,Big Dick 000,Yahoo H2H-Cat 690,categories,10,postdraft,469.l.690
7,156186,Witty Gunnars 100,Yahoo Prize Roto 156186,roto,12,postdraft,469.l.156186
8,34766,Big League Richard,Yahoo Prize H2H-Cat 34766,categories,12,predraft,469.l.34766
9,173106,Schwarbombs Pts 100,Yahoo Prize H2H-Pts 173106,points,12,postdraft,469.l.173106


# All Leagues

In [39]:
import colorsys

rosters_df = _yahoo_mod.all_rosters(leagues_df, CREDS_FILE, SEASON)
print(f'{len(rosters_df.columns)} leagues, {len(rosters_df)} roster slots')

_slot_order = ['C', '1B', '2B', '3B', 'SS', 'OF_1', 'OF_2', 'OF_3', 'Util_1', 'Util_2',
               'SP_1', 'SP_2', 'RP_1', 'RP_2', 'P_1', 'P_2', 'P_3', 'P_4',
               'BN_1', 'BN_2', 'BN_3', 'BN_4', 'BN_5', 'IL']
_ordered = [s for s in _slot_order if s in rosters_df.index]
df = rosters_df.reindex(_ordered)

# ── Color palette ─────────────────────────────────────────────────────────────
# Background: golden-ratio hue spacing × 4 lightness levels → huge combo space
# Text: 12 visually distinct dark/bold colors (black, reds, oranges, greens, etc.)
# Text cycles at a prime step (7) so it never syncs with the lightness cycle (4)

_GOLDEN      = 0.618033988749895
_BG_LIGHT    = [0.84, 0.77, 0.90, 0.71]   # 4 lightness tiers: light → darker pastel
_BG_SAT      = 0.72
_TEXT_COLORS = [
    '#111111',   # near-black
    '#b30000',   # dark red
    '#b55900',   # dark orange
    '#1a6600',   # dark green
    '#00008b',   # navy blue
    '#6a0dad',   # purple
    '#006b5e',   # dark teal
    '#8b0045',   # crimson
    '#004f80',   # dark cyan-blue
    '#4d5e00',   # olive
    '#7a1f00',   # burnt orange-red
    '#004d00',   # forest green
]

_all_players = list(dict.fromkeys(
    n for n in df.values.flatten() if not pd.isna(n) and str(n).strip()
))

_player_css = {}
for i, name in enumerate(_all_players):
    bg_h = (i * _GOLDEN) % 1
    bg_l = _BG_LIGHT[i % len(_BG_LIGHT)]
    text = _TEXT_COLORS[(i * 7) % len(_TEXT_COLORS)]   # prime step → desync from lightness
    r, g, b = colorsys.hls_to_rgb(bg_h, bg_l, _BG_SAT)
    _player_css[name] = (
        f'background-color: rgb({int(r*255)},{int(g*255)},{int(b*255)});'
        f' color: {text};'
        f' font-weight: 700'
    )

df.style.apply(
    lambda frame: frame.apply(
        lambda col: col.map(lambda v: _player_css.get(v, '') if pd.notna(v) else '')
    ),
    axis=None
)

  Yahoo Prize H2H-Cat 18397...
  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...
  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...
  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...
  Yahoo Prize Roto 156186...
  Yahoo Prize H2H-Cat 34766...
    Error: list indices must be integers or slices, not str
  Yahoo Prize H2H-Pts 173106...
9 leagues, 24 roster slots


,Mother Tuckers 100,Dr. StrangeGlove 100,Suave 100,Japanese Kokamo Okamo 100,Schlittler 100,My Bradish Burns 100,Big Dick 000,Witty Gunnars 100,Schwarbombs Pts 100
C,Yainer Diaz,Salvador Perez,Adley Rutschman,Salvador Perez,Shea Langeliers,Agustín Ramírez,Cal Raleigh,Adley Rutschman,Adley Rutschman
1B,Rafael Devers,Yandy Díaz,Christian Walker,Pete Alonso,Rafael Devers,Matt Olson,Vinnie Pasquantino,Yandy Díaz,Matt Olson
2B,Xavier Edwards,Ketel Marte,Ketel Marte,Xavier Edwards,Ketel Marte,Nico Hoerner,Ketel Marte,Xavier Edwards,Luis Arraez
3B,Matt Chapman,Eugenio Suárez,Eugenio Suárez,Kazuma Okamoto,Kazuma Okamoto,Eugenio Suárez,Maikel Garcia,Eugenio Suárez,Colson Montgomery
SS,CJ Abrams,CJ Abrams,Xavier Edwards,CJ Abrams,Geraldo Perdomo,Willy Adames,CJ Abrams,Bobby Witt Jr.,CJ Abrams
OF_1,Kyle Tucker,Pete Crow-Armstrong,Yordan Alvarez,Yordan Alvarez,Yordan Alvarez,Aaron Judge,Kyle Schwarber,Yordan Alvarez,Kyle Schwarber
OF_2,Kyle Schwarber,Seiya Suzuki,Randy Arozarena,Taylor Ward,Taylor Ward,Pete Crow-Armstrong,Yordan Alvarez,Seiya Suzuki,Julio Rodríguez
OF_3,Randy Arozarena,Taylor Ward,Kyle Stowers,Heliot Ramos,Dylan Crews,Ceddanne Rafaela,Taylor Ward,Taylor Ward,Yordan Alvarez
Util_1,Seiya Suzuki,Shohei Ohtani (Batter),Shohei Ohtani (Batter),Shohei Ohtani (Batter),Shohei Ohtani (Batter),Christian Walker,Brandon Nimmo,Gunnar Henderson,Randy Arozarena
Util_2,Taylor Ward,Alec Bohm,Dylan Crews,Yandy Díaz,Yandy Díaz,Dylan Crews,Chandler Simpson,Munetaka Murakami,Taylor Ward


In [40]:
N_PLAYERS = 3   # top batters + top pitchers to return per league

top_available = top_available_all_leagues(leagues_df, CREDS_FILE, SEASON, n=N_PLAYERS)
print(f'Done — {len(top_available)} rows ({len(leagues_df)} leagues × 2 types × {N_PLAYERS} players)')
(top_available.style
    .background_gradient(subset=['pct_owned'], cmap='Reds', vmin=0, vmax=100)
    .format({'pct_owned': '{:.0f}'})
)

  Yahoo Prize H2H-Cat 18397...
  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...
  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...
  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...
  Yahoo Prize Roto 156186...
  Yahoo Prize H2H-Cat 34766...
  Yahoo Prize H2H-Pts 173106...
Done — 54 rows (10 leagues × 2 types × 3 players)


,league,my_team,type,rank,name,team,positions,status,pct_owned
0,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,1,Kerry Carpenter,DET,OF,,66
1,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,2,Masyn Winn,STL,SS,,56
2,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,3,Trent Grisham,NYY,OF,,51
3,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,1,Quinn Priester,MIL,"SP,RP",DTD,56
4,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,2,Roki Sasaki,LAD,SP,,49
5,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,pitcher,3,Noah Cameron,KC,SP,,23
6,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,1,Heliot Ramos,SF,OF,,66
7,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,2,Masyn Winn,STL,SS,,56
8,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,3,Bryson Stott,PHI,"2B,SS",,55
9,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,pitcher,1,Sean Manaea,NYM,"SP,RP",,35


## Cross-League Player Analysis

Top available players enriched with current season stats and ATC projections.

In [ ]:
N_ANALYSIS = 3   # top batters + top pitchers to analyze per league

top5_stats = top_available_with_stats(leagues_df, CREDS_FILE, SEASON, n=N_ANALYSIS)
top5_stats

  Yahoo Prize H2H-Cat 18397...
  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...
  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...
  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...
  Yahoo Prize Roto 156186...
  Yahoo Prize H2H-Cat 34766...
  Yahoo Prize H2H-Pts 173106...
54 rows (10 leagues × 2 types × 3 players)


,league,my_team,type,rank,name,team,positions,...,proj_AVG,proj_IP,proj_W,proj_SV,proj_SO,proj_ERA,proj_WHIP
0,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,1,Kerry Carpenter,DET,OF,...,0.252951,NaN,NaN,NaN,NaN,NaN,NaN
1,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,2,Masyn Winn,STL,SS,...,0.250294,NaN,NaN,NaN,NaN,NaN,NaN
2,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,batter,3,Trent Grisham,NYY,OF,...,0.219655,NaN,NaN,NaN,NaN,NaN,NaN
3,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,1,Heliot Ramos,SF,OF,...,0.253051,NaN,NaN,NaN,NaN,NaN,NaN
4,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,2,Masyn Winn,STL,SS,...,0.250294,NaN,NaN,NaN,NaN,NaN,NaN
5,Yahoo Prize H2H-Cat 41036,Dr. StrangeGlove 100,batter,3,Bryson Stott,PHI,"2B,SS",...,0.255424,NaN,NaN,NaN,NaN,NaN,NaN
6,Yahoo Prize Roto 15981,Suave 100,batter,1,Masyn Winn,STL,SS,...,0.250294,NaN,NaN,NaN,NaN,NaN,NaN
7,Yahoo Prize Roto 15981,Suave 100,batter,2,Carlos Correa,HOU,"3B,SS",...,0.263620,NaN,NaN,NaN,NaN,NaN,NaN
8,Yahoo Prize Roto 15981,Suave 100,batter,3,Andrew Vaughn,MIL,1B,...,0.250692,NaN,NaN,NaN,NaN,NaN,NaN
9,Yahoo Prize H2H-Cat 68331,Japanese Kokamo Okamo 100,batter,1,Ezequiel Tovar,COL,SS,...,0.262220,NaN,NaN,NaN,NaN,NaN,NaN


## Upgrade Candidates

Available players who outperform at least one of your current roster players (current or projected roto score).

In [42]:
PROJ_SYSTEM = 'steamerr'   # projection system for upgrade comparison
                           # options: 'steamerr' (rest-of-season), 'atc', 'zips', 'thebat', 'thebatx'

upgrades = upgrade_candidates(leagues_df, CREDS_FILE, SEASON, n=N_ANALYSIS, proj_system=PROJ_SYSTEM)
print(f'{len(upgrades)} upgrade opportunities found (proj: {PROJ_SYSTEM})')

if upgrades.empty:
    print('No upgrades found.')
else:
    grad_cols = [c for c in ['curr_improvement', 'proj_improvement'] if c in upgrades.columns]
    fmt = {k: v for k, v in {
        'avail_curr_score':    '{:.2f}',
        'avail_proj_score':    '{:.2f}',
        'rostered_curr_score': '{:.2f}',
        'rostered_proj_score': '{:.2f}',
        'curr_improvement':    '{:.2f}',
        'proj_improvement':    '{:.2f}',
        'avail_pct_owned':     '{:.0f}',
    }.items() if k in upgrades.columns}
    styled = upgrades.style.format(fmt, na_rep='—')
    if grad_cols:
        styled = styled.background_gradient(subset=grad_cols, cmap='Greens')
    display(styled)

  Yahoo Prize H2H-Cat 18397...
  Yahoo Prize H2H-Cat 41036...
  Yahoo Prize Roto 15981...
  Yahoo Prize H2H-Cat 68331...
  Yahoo Prize H2H-Cat 148341...
  Yahoo Prize H2H-Cat 3924...
  Yahoo H2H-Cat 690...
  Yahoo Prize Roto 156186...
  Yahoo Prize H2H-Cat 34766...
  Yahoo Prize H2H-Pts 173106...
  'steamerr' projections unavailable; falling back to atc.
  Roster error (Yahoo Prize H2H-Cat 34766): list indices must be integers or slices, not str
116 upgrade opportunities found (proj: steamerr)


,league,my_team,available,avail_curr_score,avail_proj_score,avail_pct_owned,rostered,rostered_curr_score,rostered_proj_score,rostered_slot,curr_improvement,proj_improvement
0,Yahoo Prize H2H-Cat 3924,My Bradish Burns 100,José Soriano,—,6.38,18,Connelly Early,—,0.95,BN,—,5.43
1,Yahoo Prize H2H-Cat 148341,Schlittler 100,David Peterson,—,4.79,18,Andrew Painter,—,-0.60,P,—,5.39
2,Yahoo Prize Roto 15981,Suave 100,Sean Manaea,—,4.69,35,Andrew Painter,—,-0.60,BN,—,5.29
3,Yahoo Prize Roto 156186,Witty Gunnars 100,Sean Manaea,—,4.69,35,Andrew Painter,—,-0.60,P,—,5.29
4,Yahoo Prize Roto 15981,Suave 100,Bryce Miller,—,4.42,35,Andrew Painter,—,-0.60,BN,—,5.02
5,Yahoo Prize Roto 15981,Suave 100,Casey Mize,—,4.42,24,Andrew Painter,—,-0.60,BN,—,5.02
6,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,Quinn Priester,—,3.64,56,Andrew Painter,—,-0.60,BN,—,4.24
7,Yahoo Prize Roto 156186,Witty Gunnars 100,Quinn Priester,—,3.64,56,Andrew Painter,—,-0.60,P,—,4.24
8,Yahoo Prize H2H-Cat 18397,Mother Tuckers 100,Noah Cameron,—,3.44,23,Andrew Painter,—,-0.60,BN,—,4.04
9,Yahoo Prize H2H-Cat 148341,Schlittler 100,Noah Cameron,—,3.44,23,Andrew Painter,—,-0.60,P,—,4.04


## Section 3: Config — set your league ID

In [43]:
# ── Set this from the output above ───────────────────────────────────────────
LEAGUE_ID = leagues_df['league_id'].iloc[0]   # auto-picks first league; change if you have multiple
# LEAGUE_ID = '12345'                          # or hardcode here
# ─────────────────────────────────────────────────────────────────────────────

print(f'League: {leagues_df.loc[leagues_df.league_id == LEAGUE_ID, "name"].iloc[0]}')
print(f'ID:     {LEAGUE_ID}')

League: Yahoo Prize H2H-Cat 18397
ID:     18397


## Section 4: Initialize Yahoo Client

In [44]:
yf = Yahoo(league_id=LEAGUE_ID, season=SEASON, creds_file=CREDS_FILE).fetch()
print(yf)

Yahoo(league=469.l.18397, team=469.l.18397.t.9)


## Section 5: Your Team

In [45]:
# Current roster
roster = yf.roster
print(f'Roster ({len(roster)} players):')
roster

Roster (23 players):


,slot,name,team,positions,status,player_key
0,1B,Rafael Devers,SF,1B,,469.p.10235
1,2B,Xavier Edwards,MIA,"2B,SS",,469.p.11375
2,3B,Matt Chapman,SF,3B,,469.p.10205
3,BN,Victor Scott II,STL,OF,,469.p.61888
4,BN,Michael Wacha,KC,SP,,469.p.9329
5,BN,Andrew Painter,PHI,SP,,469.p.60592
6,BN,Sean Manaea,NYM,"SP,RP",,469.p.9582
7,BN,Will Warren,NYY,SP,,469.p.60295
8,C,Yainer Diaz,HOU,C,,469.p.12435
9,OF,Kyle Tucker,LAD,OF,,469.p.10480


In [46]:
# Current matchup
m = yf.matchup
if m:
    print(f"Week {m['week']}  ({m['start']} \u2192 {m['end']})")
    print(f"  You:      {m.get('you', '\u2014')}")
    print(f"  Opponent: {m.get('opponent', '\u2014')}")
else:
    print('No active matchup found (offseason or between weeks)')

Week 1  (2026-03-25 → 2026-03-29)
  You:      Mother Tuckers 100
  Opponent: $100 Standard


In [47]:
# Opponent's roster
if m:
    opp_roster = yf.opponent_roster
    print(f"Opponent roster \u2014 {m.get('opponent', '\u2014')} ({len(opp_roster)} players):")
    display(opp_roster)
else:
    print('No active matchup.')

Opponent roster — $100 Standard (23 players):


,slot,name,team,positions,status,player_key
0,1B,Vinnie Pasquantino,KC,1B,,469.p.12449
1,2B,Jose Altuve,HOU,"2B,OF",,469.p.8996
2,3B,Manny Machado,SD,3B,,469.p.9111
3,BN,Noelvi Marte,CIN,"3B,OF",,469.p.11529
4,BN,Ezequiel Tovar,COL,SS,,469.p.12322
5,BN,Brenton Doyle,COL,OF,,469.p.12136
6,BN,Jonathan Aranda,TB,1B,,469.p.12309
7,BN,Sandy Alcantara,MIA,SP,,469.p.10597
8,BN,Joe Musgrove,SD,SP,DTD,469.p.10185
9,OF,Fernando Tatis Jr.,SD,OF,,469.p.10639


## Section 6: League Standings

In [48]:
yf.standings

,rank,team,wins,losses,ties,pct,gb
0,1,Cruz’s Cool Crew,0,0,0,0.0,-
1,2,Underdog,0,0,0,0.0,-
2,3,Blizzard 💯 ⚾️,0,0,0,0.0,-
3,4,New York Yankees,0,0,0,0.0,-
4,5,$100 Standard,0,0,0,0.0,-
5,6,Benjamin Bombers,0,0,0,0.0,-
6,7,Couch Hundy,0,0,0,0.0,-
7,8,Henchmen,0,0,0,0.0,-
8,9,Mother Tuckers 100,0,0,0,0.0,-
9,10,Bronx Bombers,0,0,0,0.0,-


## Section 7: Waiver Wire & Free Agents

In [49]:
# All players on waivers (paginated — fetches every player)
waivers = yf.waivers()
print(f'Waiver wire ({len(waivers)} players):')
waivers

Waiver wire (2071 players):


,name,team,positions,status,player_key
0,Kerry Carpenter,DET,OF,,469.p.12667
1,Masyn Winn,STL,SS,,469.p.12025
2,Ramón Laureano,SD,OF,,469.p.11124
3,Andrew Vaughn,MIL,1B,,469.p.11866
4,Trent Grisham,NYY,OF,,469.p.10522
...,...,...,...,...,...
2066,Maverick Handley,BAL,C,NA,469.p.60901
2067,Bradley Blalock,MIA,SP,,469.p.63311
2068,Jack Kochanowicz,LAA,SP,,469.p.11774
2069,Germán Márquez,SD,SP,,469.p.10402


In [50]:
# All free agents (paginated — fetches every player)
# Note: many waiver-only leagues return 0 free agents; all players are on waivers instead
fa = yf.free_agents()
print(f'Free agents: {len(fa)} player(s)')
fa

Free agents: 0 player(s)


,name,team,positions,status,player_key


In [51]:
# Filter by position — change 'SP' to any Yahoo position code
# Positions: C  1B  2B  3B  SS  OF  Util  SP  RP  P
waivers_sp = yf.waivers(position='SP')
print(f'SP on waivers ({len(waivers_sp)}):')
waivers_sp

SP on waivers (637):


,name,team,positions,status,player_key
0,Noah Cameron,KC,SP,,469.p.60211
1,Quinn Priester,MIL,"SP,RP",DTD,469.p.11820
2,José Soriano,LAA,SP,,469.p.11327
3,Reynaldo López,ATL,SP,,469.p.10152
4,Bailey Ober,MIN,SP,,469.p.12096
...,...,...,...,...,...
632,Cal Quantrill,TEX,SP,NA,469.p.10573
633,Bradley Blalock,MIA,SP,,469.p.63311
634,Jack Kochanowicz,LAA,SP,,469.p.11774
635,Germán Márquez,SD,SP,,469.p.10402


## Section 8: Add / Drop Players

Uncomment and edit the player names to execute transactions.

In [52]:
# Add a free agent
# yf.add('Pete Alonso')

# Add and simultaneously drop another player
# yf.add('Pete Alonso', drop='Joey Votto')

# FAAB leagues: include a bid amount
# yf.add('Pete Alonso', drop='Joey Votto', faab=15)

# Drop only
# yf.drop('Joey Votto')

print('Uncomment a line above to execute a transaction.')

Uncomment a line above to execute a transaction.


## Section 9: Lineup Management

In [53]:
# Quick-reference: your bench and IL slots right now
bench = roster[roster['slot'].isin(['BN', 'IL', 'IL+', 'NA'])]
print('Bench / IL:')
print(bench[['slot', 'name', 'team', 'positions', 'status']].to_string(index=False))

Bench / IL:
slot            name team positions status
  BN Victor Scott II  STL        OF       
  BN   Michael Wacha   KC        SP       
  BN  Andrew Painter  PHI        SP       
  BN     Sean Manaea  NYM     SP,RP       
  BN     Will Warren  NYY        SP       


In [54]:
# Move a player to a specific slot
# yf.move('Yainer Diaz', 'BN')

# Bench a starter
# yf.bench('Cody Bellinger')

# Move a player to IL
# yf.il('Spencer Strider')

# Activate a bench player (moves to first open eligible slot)
# yf.start('Cody Bellinger')

# Swap two players' slots (e.g. start one, bench the other)
# yf.swap('Bench Player', 'Active Player')

print('Uncomment a line above to change your lineup.')

Uncomment a line above to change your lineup.


In [55]:
# Confirm updated roster after changes
# yf.roster

## Section 10: Trade Proposals

Search for players on other teams before proposing. The `team` argument matches any substring of the opponent's team name.

In [56]:
# Search for a player to check their current team / ownership
yf.search('Julio Rodriguez')

,name,team,positions,status,player_key
0,Julio Rodríguez,SEA,OF,,469.p.11384


In [57]:
# Propose a trade
# yf.trade(
#     give=['Your Player Name'],
#     receive=['Their Player Name'],
#     team='Opponent Team Name',     # case-insensitive substring of their team name
# )

# Multi-player trade
# yf.trade(
#     give=['Player A', 'Player B'],
#     receive=['Player C'],
#     team='Rival Squad',
# )

print('Uncomment above to propose a trade.')

Uncomment above to propose a trade.
